# 🟠 สมุดบันทึกที่ 3: การสร้างและประเมินผลโมเดล Naive Bayes (คนที่ 2)
**วิชา:** การทำเหมืองข้อมูล (Data Mining)  
**เนื้อหาอ้างอิง:** บทที่ 6 การจำแนกประเภท (ต่อ): K-Nearest Neighbors (KNN) และ Naive Bayes  
**ผู้รับผิดชอบ:** คนที่ 2 (Naive Bayes Specialist)  
**เป้าหมาย:** เลือกชนิด Naive Bayes ให้เหมาะสมกับลักษณะข้อมูล พร้อมให้เหตุผลทางทฤษฎี, เทรนโมเดล, วิเคราะห์หา Feature ที่ส่งผลต่อการทำนาย, และประเมินผลประสิทธิภาพ


## 1. การเลือกชนิดของ Naive Bayes (หัวใจสำคัญตามโจทย์)
ตามบทที่ 6.2.5 ของอาจารย์ มี Naive Bayes หลักๆ 3 ประเภท:
1. **Gaussian NB:** เหมาะกับแอตทริบิวต์ต่อเนื่อง (Continuous) ที่มีการแจกแจงแบบปกติ (Normal Distribution)
2. **Multinomial NB:** เหมาะกับข้อมูลนับความถี่ (Word Count / Text Classification)
3. **Bernoulli NB / Categorical NB:** เหมาะกับข้อมูลแบบ Binary หรือ หมวดหมู่ที่ไม่ต่อเนื่อง

### 💡 เหตุผลในการเลือกใช้ `GaussianNB`:
* ชุดข้อมูลมีตัวแปรตัวเลขต่อเนื่องสำคัญ ได้แก่ `absences` (จำนวนวันขาดเรียน 0-32 วัน) และ `age` (อายุ 15-22 ปี)
* หากใช้ `CategoricalNB` ตรงๆ จะเกิดปัญหา **Index Out of Bounds Error** เมื่อชุดทดสอบมีค่าตัวเลขที่ชุดฝึกสอนไม่เคยพบ (Unseen Values)
* ในทางปฏิบัติของ Scikit-learn การใช้ `GaussianNB` บนเวกเตอร์ตัวเลขและตัวแปร One-Hot Encoding เป็นวิธีที่เสถียรที่สุด ไม่ต้องเสียเวลาทำ Binning และไม่พังเมื่อเจอตัวเลขใหม่


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

# โหลดชุดข้อมูลที่เตรียมไว้จาก Notebook 1
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

X_train = train_df.drop(columns=['passed'])
y_train = train_df['passed']
X_test = test_df.drop(columns=['passed'])
y_test = test_df['passed']

print(f"📦 โหลดข้อมูลสำเร็จ: X_train = {X_train.shape}, X_test = {X_test.shape}")


## 2. การเทรนโมเดล Gaussian Naive Bayes


In [ ]:
# สร้างและฝึกสอนโมเดล GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

train_acc = gnb.score(X_train, y_train)
test_acc = gnb.score(X_test, y_test)

print("=" * 60)
print("🟠 ผลการประเมินเบื้องต้นของ Gaussian Naive Bayes:")
print("=" * 60)
print(f"• Training Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"• Testing Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)")


## 3. การวิเคราะห์หา Feature ที่ส่งผลต่อการทำนาย
เนื่องจาก Naive Bayes ไม่มี `feature_importances_` ในตัวเหมือน Decision Tree เราจึงใช้วิธีตามหลักวิทยาศาสตร์ 2 วิธี:
1. **Permutation Feature Importance:** สลับค่าข้อมูลทีละคอลัมน์แล้วดูว่าค่าความแม่นยำตกไปเท่าไหร่
2. **Class Mean Difference ($\theta$):** เปรียบเทียบค่าเฉลี่ยของแต่ละตัวแปรระหว่างกลุ่มผ่าน vs กลุ่มตก


In [ ]:
# วิธีที่ 1: Permutation Importance
perm = permutation_importance(gnb, X_test, y_test, n_repeats=10, random_state=42)
nb_importance = pd.Series(perm.importances_mean, index=X_train.columns).sort_values(ascending=False)

print("📌 Top 5 Features (Permutation Importance):")
print("-" * 50)
print(nb_importance.head(5))

# วิธีที่ 2: วิเคราะห์ค่า Mean (Theta) แยกตามคลาส Pass (1) vs Fail (0)
theta_df = pd.DataFrame(gnb.theta_, index=['Fail (0)', 'Pass (1)'], columns=X_train.columns)
mean_diff = (theta_df.loc['Pass (1)'] - theta_df.loc['Fail (0)']).abs().sort_values(ascending=False)
print("\n📌 Top 5 Features ที่มีค่าเฉลี่ยต่างกันมากที่สุดระหว่าง Pass กับ Fail:")
print("-" * 50)
print(mean_diff.head(5))


## 4. การประเมินผลผ่าน Confusion Matrix และ Metrics


In [ ]:
y_pred_nb = gnb.predict(X_test)
cm_nb = confusion_matrix(y_test, y_pred_nb)

print("📊 Confusion Matrix (Naive Bayes):")
print(cm_nb)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Fail (0)', 'Pass (1)'],
            yticklabels=['Fail (0)', 'Pass (1)'])
plt.title('Confusion Matrix: Gaussian Naive Bayes')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_nb, target_names=['Fail (0)', 'Pass (1)']))


### 💡 ประเด็นอภิปรายสำคัญของ Naive Bayes:
* แม้ Accuracy รวมจะอยู่ที่ **63.85%** (ต่ำกว่า Decision Tree) เพราะสมมติฐานความอิสระ (Conditional Independence) ถูกละเมิด
* แต่สังเกตว่า **จับเด็กตกได้ถึง 10 คนจาก 20 คน (Recall = 50.00%)** ซึ่งสูงกว่า Decision Tree แบบ Default ถึงเท่าตัว!
